# Lesson 10, one call at a time

Lesson 10 wrote a convolution in eleven lines, chose kernels by hand, checked that a
convolution moves with its input, pooled the response down to one number, built a
network of 1,537 parameters, moved a defect where none had been seen, and borrowed
weights from one task for another. This notebook takes each step on its own: on a
5×5 image or a 4×4 map where a pencil can follow it, and on the lesson's own wafer
dies where the slides' numbers come from, so that the slide's number and the call
that makes it sit side by side.

**How to use it.** Each section is one *atom*. Before each code cell that teaches
something there is a question under *Your prediction, before you run the next cell*:
answer it first, then run. `Consolidation/00_the_scikit_learn_api.ipynb` explains `fit`,
`predict` and `random_state`; this one assumes them, and uses Keras the way lesson 9
introduced it.

Every atom names the slide it comes from, by title, and lesson 10's notebooks treat
each idea at full scale. Nothing here trains a network: the whole notebook runs in
well under a minute.

## Setup

In [1]:
# Everything used here ships with the course image:
# pip install numpy==1.26.4 pandas==2.1.1 scikit-learn==1.3.1 scipy==1.11.3 tensorflow-cpu==2.21.0

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from scipy.signal import convolve2d, correlate2d

# Lesson 10's data, from the generator beside this notebook: 24 x 24 photographs
# of wafer dies, half carrying a scratch, graded by a station that records the
# wrong grade for a designed 2% of dies.
from wafer_data import TRUE_GRADER_ERROR_RATE, make_wafer_images

np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.width", 120)
keras.utils.set_random_seed(0)

# The lesson's contrast kernel: -8 at the centre, +1 around it, sum zero.
K = np.array([[1., 1., 1.],
              [1., -8., 1.],
              [1., 1., 1.]])

X, y, truth = make_wafer_images()                 # notebook 01's 4,000 dies
print(f"{len(X)} dies of {X.shape[1]}x{X.shape[2]} pixels, {int(truth.sum())} truly defective")

I0000 00:00:1795071030.587726 1755854 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1795071031.961775 1755854 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


4000 dies of 24x24 pixels, 2032 truly defective


## 1. One weighted sum, then every position

A convolutional layer writes down, at each position, the weighted sum of the pixels its kernel covers. Do one by hand on a 5×5 image, then let `correlate2d` and a Keras `Conv2D` layer, with its weights set to the same nine numbers, do all nine positions.

*From:* slides *The convolution, in words*; *One weighted sum, at every position*; handout section 3.1.

### Your prediction, before you run the next cell

A 5×5 image and a 3×3 kernel with no padding: how many outputs, in what shape? And will Keras give the same nine numbers as `correlate2d`?

In [2]:
image = np.array([[1, 2, 0, 1, 3],
                  [0, 1, 2, 2, 1],
                  [3, 1, 0, 1, 0],
                  [1, 0, 2, 3, 1],
                  [2, 1, 1, 0, 2]], dtype=float)
kernel = np.array([[1, 0, -1],
                   [1, 0, -1],
                   [1, 0, -1]], dtype=float)

print("the window at the top left:\n", image[0:3, 0:3])
print("its weighted sum, by hand:", np.sum(image[0:3, 0:3] * kernel))

print("\ncorrelate2d, every position:\n", correlate2d(image, kernel, mode="valid"))

conv = layers.Conv2D(1, 3, padding="valid", use_bias=False)
conv.build((None, 5, 5, 1))
conv.set_weights([kernel.reshape(3, 3, 1, 1)])           # Keras stores (rows, cols, in, out)
print("\nKeras Conv2D, the same nine weights:\n",
      conv(image.reshape(1, 5, 5, 1)).numpy()[0, :, :, 0])

the window at the top left:
 [[1. 2. 0.]
 [0. 1. 2.]
 [3. 1. 0.]]
its weighted sum, by hand: 2.0

correlate2d, every position:
 [[ 2.  0. -2.]
 [ 0. -4.  2.]
 [ 3. -2.  0.]]

Keras Conv2D, the same nine weights:
 [[ 2.  0. -2.]
 [ 0. -4.  2.]
 [ 3. -2.  0.]]


**What to look at.** The top-left window is the 3×3 block printed first; multiplying it by the kernel and adding gives 1 + 0 − 0 + 0 + 0 − 2 + 3 + 0 − 0 = 2, the top-left entry of both maps. A 5×5 image and a 3×3 kernel leave 3 × 3 = 9 positions, so the map is 3×3, and Keras's `Conv2D` with the same nine weights gives the same nine numbers. Note the weight array's shape, (3, 3, 1, 1): rows, columns, input channels, kernels.

**Try changing.** set `padding="same"` in the layer: the map becomes 5×5, and the new border entries are sums over windows that hang off the image, with zeros where the image is missing.

## 2. Keras computes cross-correlation, and when the flip matters

A true convolution flips the kernel by 180 degrees before sliding it; `scipy.signal.convolve2d` does, `correlate2d` and every deep learning library do not. For a learned kernel that changes nothing. For one written down by hand, it depends on the kernel.

*From:* slide *Strictly, this is cross-correlation*; handout section 3.1.

### Your prediction, before you run the next cell

For the vertical edge kernel of atom 1, and for the contrast kernel `K`, will the true convolution give the same map as Keras? Which one changes, and how?

In [3]:
def keras_map(img, ker):
    layer = layers.Conv2D(1, 3, padding="valid", use_bias=False)
    layer.build((None, *img.shape, 1))
    layer.set_weights([ker.reshape(3, 3, 1, 1)])
    return layer(img.reshape(1, *img.shape, 1)).numpy()[0, :, :, 0]

for name, ker in (("vertical edge", kernel), ("contrast K", K)):
    flipped = np.rot90(ker, 2)
    print(f"{name}: unchanged by the 180-degree flip? {np.array_equal(flipped, ker)}")
    print("  Keras = correlate2d:   ", np.allclose(keras_map(image, ker), correlate2d(image, ker, mode="valid")))
    print("  Keras = convolve2d:    ", np.allclose(keras_map(image, ker), convolve2d(image, ker, mode="valid")))
    print("  Keras = -convolve2d:   ", np.allclose(keras_map(image, ker), -convolve2d(image, ker, mode="valid")))

vertical edge: unchanged by the 180-degree flip? False
  Keras = correlate2d:    True
  Keras = convolve2d:     False
  Keras = -convolve2d:    True
contrast K: unchanged by the 180-degree flip? True
  Keras = correlate2d:    True
  Keras = convolve2d:     True
  Keras = -convolve2d:    False


**What to look at.** Keras equals `correlate2d` for both kernels: it never flips. The true convolution `convolve2d` agrees with it on the contrast kernel `K`, which a 180-degree rotation leaves unchanged, and gives exactly the negative on the vertical edge kernel, which the rotation turns into its opposite. For a learned kernel this changes nothing, because training would simply learn the flipped weights. For a kernel written by hand it is the difference between "brighter on the left" and "brighter on the right".

**Try changing.** use the horizontal edge kernel `kernel.T`: it is not symmetric either, and the same three lines tell you so.

## 3. Output sizes, from the formula and from Keras

The output size along one side is ⌊(H + 2p − f)/s⌋ + 1, with H the input's side, f the kernel, p the padding and s the stride. Keras does not take p: it takes `padding="valid"` (p = 0) or `"same"`. Build each layer on a 24×24 input and read the shape it reports.

*From:* slides *How big is the output?*; *Worked on this lesson's 24×24 images*; *Two cases cover almost everything*; handout section 3.2.

### Your prediction, before you run the next cell

A 3×3 kernel with stride 2 and **no** padding on a 24×24 image: 11 or 12? And with `padding="same"`?

In [4]:
rows = []
for f, padding, s in ((3, "valid", 1), (3, "same", 1), (5, "valid", 1),
                      (5, "same", 1), (3, "same", 2), (3, "valid", 2)):
    p = (f - 1) // 2 if padding == "same" else 0
    H = 24
    formula = (H + 2 * p - f) // s + 1
    model = keras.Sequential([keras.Input((H, H, 1)),
                              layers.Conv2D(1, f, strides=s, padding=padding)])
    rows.append({"kernel f": f, "padding": f"{padding} (p={p})", "stride s": s,
                 "formula": formula, "Keras": model.output_shape[1]})
pd.DataFrame(rows)

,kernel f,padding,stride s,formula,Keras
0,3,valid (p=0),1,22,22
1,3,same (p=1),1,24,24
2,5,valid (p=0),1,20,20
3,5,same (p=2),1,24,24
4,3,same (p=1),2,12,12
5,3,valid (p=0),2,11,11


**What to look at.** The formula and Keras agree in all six rows. `"same"` is p = (f − 1)/2, one pixel for a 3×3 kernel and two for a 5×5, and with stride 1 it keeps 24. Stride 2 with padding gives ⌊(24 + 2 − 3)/2⌋ + 1 = 11 + 1 = 12; without padding ⌊(24 − 3)/2⌋ + 1 = 10 + 1 = 11. The floor drops the last window that would hang off the edge, and Keras raises no error about it.

**Try changing.** try f = 2, stride 2, `"valid"`: (24 − 2)/2 + 1 = 12, the shape pooling gives.

## 4. A zero sum cancels a constant, and only suppresses a curve

`K`'s nine weights sum to zero, so on a patch where every pixel has the same value v the response is v × 0 = 0. The wafer's background is not constant, it varies smoothly. Try four 5×5 patches: constant, a straight ramp, a gentle curve, and a flat patch with one dark pixel.

*From:* slide *A constant patch gives exactly zero*; handout section 3.3.

### Your prediction, before you run the next cell

Which of the four patches give exactly zero? Which gives the largest response?

In [5]:
r, c = np.mgrid[0:5, 0:5].astype(float)
patches = {
    "constant 0.6":            np.full((5, 5), 0.6),
    "straight ramp":           0.5 + 0.02 * r + 0.01 * c,
    "gentle curve":            0.5 + 0.01 * (r - 2) ** 2,
    "flat, one dark pixel":    np.where((r == 2) & (c == 2), 0.26, 0.6),
}
for name, patch in patches.items():
    centre = correlate2d(patch, K, mode="valid")[1, 1]     # the window centred on (2, 2)
    print(f"{name:<24s} response at the centre {centre:+.3f}")

constant 0.6             response at the centre +0.000
straight ramp            response at the centre -0.000
gentle curve             response at the centre +0.060
flat, one dark pixel     response at the centre +2.720


**What to look at.** The constant patch gives exactly 0: every pixel is 0.6, and 0.6 × (8 − 8) = 0. The straight ramp gives 0 too, because `K` is symmetric and the rise on one side of the centre cancels the fall on the other. The gentle curve does not cancel: its response is 0.060, small but not zero, which is what "suppressed rather than cancelled" means for the wafer's film thickness. One dark pixel 0.34 below a flat 0.6 gives 8 × 0.34 = 2.720, forty-five times the curve's response.

**Try changing.** make the curve steeper, 0.05 instead of 0.01: the response grows to 0.300, and a background that curves fast enough would start to look like a defect.

## 5. The sign picks dark or bright: 3.129 against 1.118, and the flip

On the lesson's first defective and first clean die, the contrast kernel's strongest response is 3.129 against 1.118. Flip every sign — +8 at the centre, −1 around it — and the sum is still zero.

*From:* slide *Nine numbers, no training: 3.129 against 1.118*; handout section 3.3.

### Your prediction, before you run the next cell

With the signs flipped, where on the defective die will the strongest response be: on the scratch, beside it, or somewhere else?

In [6]:
defective = X[truth == 1][0]
clean = X[truth == 0][0]

def strongest(img, ker):
    response = correlate2d(np.pad(img, 1), ker, mode="valid")   # zero padding, as in notebook 01
    return response.max(), np.unravel_index(response.argmax(), response.shape)

for name, ker in (("-8 centre (K)", K), ("+8 centre (-K)", -K)):
    d, where = strongest(defective, ker)
    cl, where_clean = strongest(clean, ker)
    print(f"{name}, zero padding: defective {d:.3f} at {where}, clean {cl:.3f} at {where_clean}")

# The same flipped kernel without padding, so the border cannot take part:
print(f"+8 centre (-K), no padding: defective {correlate2d(defective, -K, mode='valid').max():.3f}, "
      f"clean {correlate2d(clean, -K, mode='valid').max():.3f}")

-8 centre (K), zero padding: defective 3.129 at (8, 3), clean 1.118 at (21, 8)
+8 centre (-K), zero padding: defective 2.848 at (23, 0), clean 3.390 at (23, 23)
+8 centre (-K), no padding: defective 2.428, clean 1.061


**What to look at.** With −8 at the centre the strongest response, 3.129, sits on the scratch at row 8, column 3, against 1.118 on the clean die. With the signs flipped and zero padding, both dies peak in a **corner**: (23, 0) on the defective die and (23, 23) on the clean one. A die meets the padding's zeros there, and a bright pixel surrounded by zeros is exactly what the flipped kernel looks for. Without padding, the flipped kernel finds the brighter rim beside the scratch, 2.428 against 1.061: it still separates these two dies, but less strongly than the right sign does. The zero sum removes the background; the sign decides what is reported.

**Try changing.** look at where the flipped kernel peaks without padding, with `np.unravel_index`: it is next to row 8, column 3, not on it.

## 6. Two dies are a demonstration: a threshold from many

3.129 against 1.118 suggests a threshold in between. A threshold is a model with one parameter, so choose it on some dies and score it on others: the first 2,000 of the 4,000 to choose, the last 2,000 to score.

*From:* slide *Nine numbers, no training: 3.129 against 1.118*; handout section 3.3; lesson 5's held-out set.

### Your prediction, before you run the next cell

Will the best threshold sit near the midpoint of 1.118 and 3.129, about 2.1? And will it reach the ceiling of the dies it is scored on?

In [7]:
peak = np.array([correlate2d(np.pad(img, 1), K, mode="valid").max() for img in X])
choose, score = slice(0, 2000), slice(2000, 4000)

candidates = np.linspace(1.0, 3.5, 251)
accuracy = [np.mean((peak[choose] > t) == y[choose]) for t in candidates]
best = candidates[int(np.argmax(accuracy))]

print(f"peak response, clean dies:     median {np.median(peak[truth == 0]):.3f}, max {peak[truth == 0].max():.3f}")
print(f"peak response, defective dies: median {np.median(peak[truth == 1]):.3f}, min {peak[truth == 1].min():.3f}")
print(f"\nthreshold chosen on the first 2,000: {best:.2f}")
print(f"on the last 2,000, against the recorded grade: {np.mean((peak[score] > best) == y[score]):.4f}")
print(f"the ceiling of those 2,000 (recorded = true):   {np.mean(y[score] == truth[score]):.4f}")

peak response, clean dies:     median 1.181, max 1.874
peak response, defective dies: median 3.456, min 2.269

threshold chosen on the first 2,000: 1.88
on the last 2,000, against the recorded grade: 0.9765
the ceiling of those 2,000 (recorded = true):   0.9765


**What to look at.** Over 4,000 dies the two classes do not overlap at all: the clean dies' peaks stop at 1.874 and the defective dies' start at 2.269. The threshold chosen on the first 2,000, 1.88, sits just above the largest clean peak rather than at the midpoint of the first two dies. Scored on the other 2,000, it reaches 0.9765, which is exactly the ceiling of those 2,000: it agrees with the true grade on every one of them, and every miss is a grade the station recorded wrongly.

**Try changing.** choose the threshold on 20 dies, `slice(0, 20)`, and score it on the same last 2,000.

## 7. Shift, then convolve, is convolve, then shift: away from the border

Equivariance: shifting the image and then convolving gives the same map as convolving and then shifting. Notebook 01 checks it on the interior only. Check the whole map too, and see where the two orders disagree.

*From:* slides *Shift the input, and the response shifts with it*; *Shift, then convolve, is convolve, then shift*; handout section 4.1.

### Your prediction, before you run the next cell

Shift the die three columns to the right with `np.roll`, which wraps the last columns round to the front. Will the two orders agree everywhere, or only somewhere?

In [8]:
def response(img):
    return correlate2d(np.pad(img, 1), K, mode="valid")      # same size as the image

shifted_first = response(np.roll(defective, 3, axis=1))
shifted_after = np.roll(response(defective), 3, axis=1)
gap = np.abs(shifted_first - shifted_after)

print(f"largest disagreement, whole 24x24 map: {gap.max():.3f}")
print(f"largest disagreement, columns 5 to 18: {gap[:, 5:19].max():.3f}")
print("columns where the orders disagree:", np.where(gap.max(axis=0) > 1e-12)[0])

largest disagreement, whole 24x24 map: 1.725
largest disagreement, columns 5 to 18: 0.000
columns where the orders disagree: [ 0  2  3 23]


**What to look at.** In columns 5 to 18 the two orders agree exactly, 0.000. Over the whole map they disagree by up to 1.725, in columns 0, 2, 3 and 23. `np.roll` wraps the last three columns round to the front, so the shifted image has a seam that the convolution sees, and the zero padding is applied to a different image in each order. Equivariance is a statement about the interior; at the border it depends on the padding and on how the shift is done, which is why notebook 01 compares the interior only.

**Try changing.** shift by 6 columns: the band of disagreement widens with the shift.

## 8. Pooling by hand: one pixel of shift, inside a window and across its edge

`MaxPooling2D(2)` keeps the maximum of each 2×2 window and discards where it was. On a 4×4 map with one strong response, move that response by one pixel twice: once inside its window, once across the window's edge. Then pool the whole map with `GlobalMaxPooling2D`.

*From:* slides *Pooling: from where to whether*; *Three things pooling does*; handout section 4.2.

### Your prediction, before you run the next cell

Which of the two one-pixel shifts leaves the 2×2-pooled output identical? And the global maximum?

In [9]:
def feature_map(row, col):
    m = np.full((4, 4), 0.1)
    m[row, col] = 3.0
    return m.reshape(1, 4, 4, 1)

pool, gpool = layers.MaxPooling2D(2), layers.GlobalMaxPooling2D()
for label, (row, col) in (("response at (1, 0)", (1, 0)),
                          ("shifted to (1, 1), same window", (1, 1)),
                          ("shifted to (1, 2), next window", (1, 2))):
    m = feature_map(row, col)
    print(f"{label:<32s} pooled 2x2: {pool(m).numpy()[0, :, :, 0].ravel()}  "
          f"global: {gpool(m).numpy().ravel()}")

response at (1, 0)               pooled 2x2: [3.  0.1 0.1 0.1]  global: [3.]
shifted to (1, 1), same window   pooled 2x2: [3.  0.1 0.1 0.1]  global: [3.]
shifted to (1, 2), next window   pooled 2x2: [0.1 3.  0.1 0.1]  global: [3.]


**What to look at.** Moving the response from (1, 0) to (1, 1) keeps it inside the same 2×2 window, and the pooled output is identical: [3.0, 0.1, 0.1, 0.1]. Moving it to (1, 2) carries it across the window's edge, and the 3.0 moves to the second pooled cell. One pixel of shift, two different outcomes: local pooling gives *approximate* invariance. The global maximum is 3.0 in all three cases, because it has no windows to cross.

**Try changing.** move the response to (2, 2): which pooled cell does it land in?

## 9. The receptive field, counted: 8×8

A 3×3 kernel after a 2×2 pooling, on top of the first 3×3 layer, sees an 8×8 window of the original image. Count it: build the two layers and the pooling with every weight 1, ask which input pixels can change one output, and count them. Average pooling stands in for max pooling, because a maximum passes a gradient to one pixel only; the window each output can see is the same.

*From:* slide *Three things pooling does*; handout section 4.2.

### Your prediction, before you run the next cell

How many of the 576 input pixels can change the output at the centre of the 12×12 map: 9, 36 or 64?

In [10]:
probe = keras.Sequential([keras.Input((24, 24, 1)),
                          layers.Conv2D(1, 3, padding="same", use_bias=False, kernel_initializer="ones"),
                          layers.AveragePooling2D(2),
                          layers.Conv2D(1, 3, padding="same", use_bias=False, kernel_initializer="ones")])
inputs = tf.Variable(np.random.default_rng(0).uniform(0.4, 0.6, (1, 24, 24, 1)))
with tf.GradientTape() as tape:
    out = probe(inputs)[0, 6, 6, 0]               # one output, in the middle of the 12x12 map
grad = tape.gradient(out, inputs).numpy()[0, :, :, 0]
seen = np.argwhere(grad != 0)

print(f"input pixels that can change it: {len(seen)}")
print(f"rows {seen[:, 0].min()}-{seen[:, 0].max()}, columns {seen[:, 1].min()}-{seen[:, 1].max()}")

input pixels that can change it: 64
rows 9-16, columns 9-16


**What to look at.** Exactly 64 input pixels can change the centre output, rows 9 to 16 and columns 9 to 16: an 8×8 window. The second 3×3 kernel spans 3 pooled cells, each covering 2 pixels of the first layer's map, so 6 × 6 of it; the first 3×3 layer adds one pixel on each side, 6 + 2 = 8. Nine weights in the second layer, an 8×8 view of the die.

**Try changing.** add a second `AveragePooling2D(2)` and `Conv2D` on top, and count again for the centre of the 6×6 map.

## 10. Counting parameters: 80, 1,168, and 147,712

A kernel's depth always equals the number of input channels. Build the lesson's network and a 256-unit dense layer on the 576 pixels, and read each layer's `count_params()` and the shape of its weight array.

*From:* slides *80 parameters against 147,712*; *1,537 parameters, in full*; *The count students get wrong: 1,168*; handout sections 2 and 5.

### Your prediction, before you run the next cell

The second convolutional layer has 16 kernels of 3×3 on 8 input channels: 160 parameters, or more?

In [11]:
inputs = keras.Input((24, 24, 1))
x = layers.Conv2D(8, 3, activation="relu", padding="same", name="c1")(inputs)
x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(16, 3, activation="relu", padding="same", name="c2")(x)
x = layers.MaxPooling2D(2)(x)
x = layers.GlobalMaxPooling2D()(x)
x = layers.Dense(16, activation="relu")(x)
net = keras.Model(inputs, layers.Dense(1, activation="sigmoid")(x))

for layer in net.layers:
    shapes = [w.shape for w in layer.get_weights()]
    if shapes:
        print(f"{layer.name:<8s} weights {str(shapes[0]):<16s} biases {str(shapes[1]):<6s} -> {layer.count_params():>5,}")
print(f"total: {net.count_params():,}")

dense = keras.Sequential([keras.Input((576,)), layers.Dense(256)])
print(f"\none dense layer of 256 on 576 pixels: {dense.count_params():,} = 576 x 256 + 256")

c1       weights (3, 3, 1, 8)     biases (8,)   ->    80
c2       weights (3, 3, 8, 16)    biases (16,)  -> 1,168
dense    weights (16, 16)         biases (16,)  ->   272
dense_1  weights (16, 1)          biases (1,)   ->    17
total: 1,537

one dense layer of 256 on 576 pixels: 147,712 = 576 x 256 + 256


**What to look at.** `c1`'s weights have shape (3, 3, 1, 8): 72 weights plus 8 biases, 80 parameters. `c2`'s are (3, 3, **8**, 16): each of its 16 kernels is 3 × 3 across all 8 input channels, so 16 × 72 + 16 = 1,168, not the 16 × 9 + 16 = 160 that forgetting the channels gives. The head is 272 + 17 = 289, and the total 1,537. One dense layer of 256 units on the 576 pixels is already 147,712.

**Try changing.** set the first layer to 32 kernels: how many parameters does `c2` have now, and why did it change when `c2` did not?

## 11. The ceiling, realised on the set it is measured on

The station flips each grade with probability 0.02, so a model that knows every true grade scores 0.98 *on average*. A finite batch draws its own fraction. Count it on notebook 01's 4,000 dies and on notebook 02's 2,000 test dies, and compare the first with the binomial spread.

*From:* slides *The grading station is wrong 2% of the time*; *Head to head, and where the missing 2% lives*; handout sections 1.1 and 5.1.

### Your prediction, before you run the next cell

Will both batches have exactly 2% of their grades wrong? If not, how far from 2% is still ordinary?

In [12]:
X_test, y_test, t_test = make_wafer_images(seed=12345, n_images=2000)   # notebook 02's test set
e = TRUE_GRADER_ERROR_RATE

for name, labels, true in (("notebook 01, 4,000 dies", y, truth),
                           ("notebook 02 test, 2,000 dies", y_test, t_test)):
    wrong = np.mean(labels != true)
    sd = np.sqrt(e * (1 - e) / len(labels))
    print(f"{name}: {int((labels != true).sum())} wrong = {wrong:.4f}, "
          f"ceiling {1 - wrong:.4f}, {(wrong - e) / sd:+.1f} binomial sd from 0.02")

q = 1.0                                           # agreement with the truth
print(f"\nq(1-e) + (1-q)e at q = 1, e = 0.02: {q * (1 - e) + (1 - q) * e:.4f}")

notebook 01, 4,000 dies: 102 wrong = 0.0255, ceiling 0.9745, +2.5 binomial sd from 0.02
notebook 02 test, 2,000 dies: 40 wrong = 0.0200, ceiling 0.9800, +0.0 binomial sd from 0.02

q(1-e) + (1-q)e at q = 1, e = 0.02: 0.9800


**What to look at.** Notebook 01's 4,000 dies drew 102 wrong grades, 0.0255, so their ceiling is 0.9745; with a binomial standard deviation of √(0.02 × 0.98 / 4,000) = 0.0022 that is 2.5 standard deviations high, an unusual draw and not a bug. Notebook 02's 2,000 test dies drew exactly 40, 0.0200, so their ceiling is 0.9800. And q(1 − e) + (1 − q)e at q = 1 gives 0.98: a model right about every die scores the station's agreement with the truth, no more.

**Try changing.** set q = 0.95 in the last line: the formula predicts 0.932.

## 12. For a dense layer, a permutation is a relabelling

Shuffle the 576 pixel positions with one fixed permutation. For a dense layer that is only a renaming: shuffle the rows of its weight matrix the same way and its output is identical. A convolution has no rows to rename: its nine weights are tied to neighbours.

*From:* slides *What is the convolution actually using?*; *Two assumptions, and only one is about arrangement*; handout section 9.

### Your prediction, before you run the next cell

After shuffling the pixels and the dense layer's weight rows with the same permutation, will the dense outputs differ at all? And the contrast map's strongest response?

In [13]:
rng = np.random.default_rng(10003)
perm = rng.permutation(576)
pixels = defective.ravel()
W = rng.normal(0, 0.1, (576, 4))                  # a dense layer of 4 units, random weights

print("dense, original:          ", pixels @ W)
print("dense, both shuffled:     ", pixels[perm] @ W[perm])
print("largest difference:", np.abs(pixels @ W - pixels[perm] @ W[perm]).max())

scrambled = pixels[perm].reshape(24, 24)
print(f"\ncontrast kernel, strongest response: photographed {response(defective).max():.3f}, "
      f"shuffled {response(scrambled).max():.3f}")

dense, original:           [ 0.811  0.383  0.388 -0.302]
dense, both shuffled:      [ 0.811  0.383  0.388 -0.302]
largest difference: 5.551115123125783e-16

contrast kernel, strongest response: photographed 3.129, shuffled 4.156


**What to look at.** The dense layer's four outputs are identical before and after the shuffle, up to 5.6 × 10⁻¹⁶ of rounding: shuffling the pixels and the weight rows the same way is only a renaming, which is why lesson 9's dense network could not notice a permutation. The contrast kernel's strongest response changes from 3.129 to 4.156: on the shuffled die its 3×3 windows hold pixels that were never neighbours, the scratch's dark pixels are scattered, and a window can now pair one of them with eight unrelated bright pixels. A convolution reads neighbours, so a permutation changes what it reads; a trained convolution can still adapt to a fixed one, which is what section 9 measures.

**Try changing.** shuffle with a different permutation for the weights than for the pixels: the dense outputs no longer agree, which is why the permutation must be the same for every image.

## 13. Augmentation: one offset per image, not per copy

Notebook 02 adds four shifted copies of every training image, each rolled by its own random offset. Done on six dies, it is easy to see what one offset per copy would have produced instead, and that the new arrays are copies, not new photographs.

*From:* slide *Augmentation buys 8 points and leaves it 44 short*; handout section 7.

### Your prediction, before you run the next cell

Six dies, four copies each: how many arrays, how many different photographs, and how many different offsets if one offset is drawn per copy instead of per image?

In [14]:
six = X[:6]
rng = np.random.default_rng(0)

per_image = rng.integers(-10, 11, (4, 6, 2))      # copy x image x (rows, columns)
per_copy = np.repeat(rng.integers(-10, 11, (4, 1, 2)), 6, axis=1)

copies = [np.roll(np.roll(six[i], dr, axis=0), dc, axis=1)
          for k in range(4) for i, (dr, dc) in enumerate(per_image[k])]
print(f"arrays after augmentation: {len(six) + len(copies)}, from {len(six)} photographs")
print(f"different offsets, drawn per image: {len({tuple(o) for o in per_image.reshape(-1, 2)})}")
print(f"different offsets, drawn per copy:  {len({tuple(o) for o in per_copy.reshape(-1, 2)})}")
print("copy 1 of die 0 is die 0 rolled; same pixel values, sorted:",
      np.allclose(np.sort(copies[0].ravel()), np.sort(six[0].ravel())))

arrays after augmentation: 30, from 6 photographs
different offsets, drawn per image: 24
different offsets, drawn per copy:  4
copy 1 of die 0 is die 0 rolled; same pixel values, sorted: True


**What to look at.** Six dies with four copies each make 30 arrays from 6 photographs: the copies carry the same pixel values, moved. Drawn per image, the 24 copies have 24 different offsets; drawn per copy, only 4, the same four positions applied to every die, which adds four fixed special cases rather than variety. Notebook 02's 15,000 training arrays are 3,000 photographs and their shifted copies, correlated with their sources.

**Try changing.** set `max_shift` from 10 to 1, `rng.integers(-1, 2, ...)`: how many different offsets are possible at all?

## 14. Transfer: what is copied, what is new, what is frozen

Notebook 03 copies the two convolutional layers of a three-class network into a new binary one with `set_weights`, and freezes them with `trainable = False`. Do it without training anything, and read what changed.

*From:* slides *Start from someone else's weights*; *When transfer makes things worse*; handout section 10.

### Your prediction, before you run the next cell

After copying `c1` and `c2`, is the new network's output layer the old one's? And with both layers frozen, how many of its 1,537 parameters can still train?

In [15]:
def conv_net(n_out, act, seed):
    keras.utils.set_random_seed(seed)
    inputs = keras.Input((24, 24, 1))
    x = layers.Conv2D(8, 3, activation="relu", padding="same", name="c1")(inputs)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(16, 3, activation="relu", padding="same", name="c2")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.GlobalMaxPooling2D()(x)
    x = layers.Dense(16, activation="relu", name="head")(x)
    return keras.Model(inputs, layers.Dense(n_out, activation=act, name="out")(x))

source = conv_net(3, "softmax", seed=0)           # stands in for the typing network
target = conv_net(1, "sigmoid", seed=1)

for name in ("c1", "c2"):
    target.get_layer(name).set_weights(source.get_layer(name).get_weights())
    target.get_layer(name).trainable = False

same = lambda a, b: all(np.array_equal(u, v) for u, v in zip(a.get_weights(), b.get_weights()))
print("c1 copied:", same(target.get_layer("c1"), source.get_layer("c1")))
print("c2 copied:", same(target.get_layer("c2"), source.get_layer("c2")))
print("head copied:", same(target.get_layer("head"), source.get_layer("head")))
print("output shapes, source and target:", source.output_shape, target.output_shape)

trainable = sum(np.prod(w.shape) for w in target.trainable_weights)
print(f"\ntrainable with c1 and c2 frozen: {trainable:,} of {target.count_params():,}")

c1 copied: True
c2 copied: True
head copied: False
output shapes, source and target: (None, 3) (None, 1)

trainable with c1 and c2 frozen: 289 of 1,537


**What to look at.** `c1` and `c2` now hold the source's weights, and the head does not: it is the target's own, newly initialised, and the output layer has a different shape, 1 unit against 3. What transfers is the feature extractor, not the old classifier. With both convolutional layers frozen only the head's 289 parameters are trainable, of 1,537. In a real run, call `compile` again after changing `trainable`, as notebook 03 does, so the optimiser sees the change.

**Try changing.** set `trainable = True` on both layers: the count goes back to 1,537, and this is the warm start the handout recommends when in doubt.

## 15. Four hand-made numbers per die

Section 12's classical models get four numbers per die instead of 576 pixels: the maximum, the minimum, the standard deviation and the largest absolute value of the contrast kernel's response. Compute them for the two dies of atom 5.

*From:* slides *Ten lessons on one problem*; *Four hand-written numbers: within one die of the ceiling*; handout section 12.

### Your prediction, before you run the next cell

Which of the four numbers separates the clean die from the defective one most clearly?

In [16]:
def hand_features(img):
    r = correlate2d(img, K, mode="valid")        # notebook 03 uses the valid map, 22 x 22
    return {"max": r.max(), "min": r.min(), "std": r.std(), "max |r|": np.abs(r).max()}

pd.DataFrame({"defective die": hand_features(defective),
              "clean die": hand_features(clean)}).T.round(3)

,max,min,std,max |r|
defective die,3.129,-2.428,0.560,3.129
clean die,1.118,-1.061,0.408,1.118


**What to look at.** On the defective die the maximum is 3.129 against 1.118, the minimum −2.428 against −1.061, the standard deviation 0.560 against 0.408, and the largest absolute value 3.129 against 1.118. The maximum separates the two most clearly, by a factor of nearly three; the minimum is the flipped kernel's rim of atom 5 with its sign changed. Logistic regression, k-nearest neighbours and the support vector machine, given these four numbers for 3,000 dies, reach the ceiling.

**Try changing.** compute the four numbers for the 4,000 dies and look at the two classes' ranges of each, as atom 6 did for the maximum.